# Step 1 — Open Google Colab and install dependencies

In [38]:
!pip install --force-reinstall -U langchain langchain-openai langchain-community langchain-core langchain-huggingface faiss-cpu pypdf -q
import site
from importlib import reload
reload(site)
print("Environment forcefully refreshed!")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 109.4/109.4 kB 5.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.6/79.6 kB 6.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.7/57.7 kB 3.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.0/42.0 kB 3.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.9/40.9 kB 2.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.4/40.4 kB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 27.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 661.5/661.5 kB 26.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.6/16.6 MB 23.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.2/100.2 kB 5.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 472.3/472.3 kB 16.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 53.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 

# Step 3 — Set up your API key

In [27]:
import os
os.environ["OPENAI_API_KEY"] = "your-api-key-here"  # Replace before running
print("API key set!")

API key set!


# Step 4 — Create a sample document to query

In [28]:
# We'll create a text file about AI for testing
sample_text = """
Artificial Intelligence (AI) is the simulation of human intelligence
in machines programmed to think and learn like humans.

Machine Learning is a subset of AI that enables systems to learn
from data without being explicitly programmed. Common algorithms
include Linear Regression, Decision Trees, and Neural Networks.

Deep Learning is a subset of Machine Learning that uses neural
networks with many layers to analyze data. It powers applications
like image recognition and natural language processing.

Natural Language Processing (NLP) allows computers to understand,
interpret, and generate human language. Applications include
chatbots, translation, and sentiment analysis.

LangChain is a framework for developing applications powered by
large language models. It enables chaining of components like
prompts, models, and memory to build powerful AI applications.

Retrieval-Augmented Generation (RAG) is a technique that combines
information retrieval with text generation. The system first
retrieves relevant documents then uses them to generate answers.

Vector databases store data as mathematical vectors enabling
semantic similarity search. Popular options include FAISS,
Pinecone, and Chroma.
"""

with open("ai_knowledge.txt", "w") as f:
    f.write(sample_text)

print("Document created!")

Document created!


# Step 5 — Load and split the document

In [29]:
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

# Load the document
loader = TextLoader("ai_knowledge.txt")
documents = loader.load()

# Initialize the text splitter
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=100
)

# Split the document into chunks
chunks = text_splitter.split_documents(documents)

print(f"Document split into {len(chunks)} chunks")
for i, chunk in enumerate(chunks):
  print(f"\nChunk {i+1}:")
  print(chunk.page_content[:100] + "...")

Document split into 4 chunks

Chunk 1:
Artificial Intelligence (AI) is the simulation of human intelligence 
in machines programmed to thin...

Chunk 2:
Deep Learning is a subset of Machine Learning that uses neural 
networks with many layers to analyze...

Chunk 3:
LangChain is a framework for developing applications powered by 
large language models. It enables c...

Chunk 4:
Vector databases store data as mathematical vectors enabling 
semantic similarity search. Popular op...


# Step 6 — Create Vector Database (FAISS)


In [30]:
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS

# Using the updated langchain-huggingface package for local embeddings
embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")

vectorstore = FAISS.from_documents(chunks, embeddings)

print("Vector database created using HuggingFace embeddings!")
print(f"Stored {vectorstore.index.ntotal} vectors")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Vector database created using HuggingFace embeddings!
Stored 4 vectors


Step 7 — Build the Q&A Chain

In [48]:
from langchain_huggingface import HuggingFacePipeline
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser
from transformers import pipeline

# 1. Initialize a local LLM using a widely compatible task and model
print("Loading local LLM (gpt2)... this will be much faster.")
model_id = "gpt2"
# We use 'text-generation' which is supported by the environment
pipe = pipeline("text-generation", model=model_id, max_new_tokens=50, pad_token_id=50256)
llm = HuggingFacePipeline(pipeline=pipe)

# 2. Define the prompt
template = """Answer the question based only on the following context:
{context}

Question: {question}

Answer:"""
prompt = ChatPromptTemplate.from_template(template)

# 3. Initialize Retriever from existing vectorstore
retriever = vectorstore.as_retriever(search_kwargs={"k": 1})

# Helper to format documents
def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

# 4. Build the LCEL chain
qa_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

def ask_question(question):
    print(f"\n❓ Question: {question}")
    answer = qa_chain.invoke(question)
    # GPT2 outputs the prompt as well, so we clean it up if needed
    print(f"🤖 Answer: {answer.split('Answer:')[-1].strip()}")

print("Local Q&A Chain successfully built!")

Loading local LLM (gpt2)... this will be much faster.


config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/548M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

GPT2LMHeadModel LOAD REPORT from: gpt2
Key                  | Status     |  | 
---------------------+------------+--+-
h.{0...11}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


...

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Passing `generation_config` together with generation-related arguments=({'pad_token_id', 'max_new_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.


Local Q&A Chain successfully built!


# Step 8 — Ask questions!

In [50]:
# Re-running the test with the newly initialized local LCEL chain
ask_question("What is LangChain?")
ask_question("How does RAG work?")
ask_question("what is the work of AI?")


❓ Question: What is LangChain?


Both `max_new_tokens` (=50) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=50) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


🤖 Answer: LangChain is a framework for developing applications powered by 

large language models. It enables chaining of components like 

prompts, models, and memory to build powerful AI applications.

Retrieval-

❓ Question: How does RAG work?


Both `max_new_tokens` (=50) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


🤖 Answer: цостроция частить частить is a program used to determine the

best path for choosing the best language for your project.

❓ Question: what is the work of AI?
🤖 Answer: AI is challenging human decision making.

Question
